# RAG for Plastic Classification Research

In [ ]:
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Home RAG"

In [ ]:
import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

c:\Users\dorky\anaconda3\envs\LangLang\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
from langsmith import traceable

## Indexing

### Loading Documents 


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

data_folder = Path("../data")  # folder where your PDFs are stored
pdf_files = list(data_folder.glob("*.pdf"))  # all PDFs in the folder

docs = []

for pdf in pdf_files:
    loader = PyPDFLoader(str(pdf))
    docs.extend(loader.load())

print(f"Total documents loaded: {len(docs)}")
print(f"First doc preview:\n{docs[0].page_content[:500]}...")

Total documents loaded: 150
First doc preview:
Review
Microplastics in freshwaters and drinking water: Critical review and
assessment of data quality
Albert A. Koelmans a, *, Nur Hazimah Mohamed Nor a, Enya Hermsen a, Merel Kooi a,
Svenja M. Mintenig b, c, Jennifer De France d, **
a Aquatic Ecology and Water Quality Management Group, Wageningen University, the Netherlands
b Copernicus Institute of Sustainable Development, Utrecht University, the Netherlands
c KWR Watercycle Research Institute, Nieuwegein, the Netherlands
d World Health Organ...


### Splitting documents

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # adjust to fit model tokens
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)

split_docs = text_splitter.split_documents(docs)
print(f"Total chunks for RAG: {len(split_docs)}")

Total chunks for RAG: 929


### Storing documents

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = InMemoryVectorStore(embeddings)

document_ids = vector_store.add_documents(documents=split_docs)
print(document_ids[:3])

['25669d4c-6135-426b-a15e-d1ebe310e4c8', '3aecd76d-4634-4200-a809-ccf5cd03de20', 'a90d1bc2-1b09-4c8f-9bf5-f658021de620']


## Retrieval and generation

In [7]:
# create a retriever
retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

In [11]:
# format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# create prompt template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}
""")

In [13]:
# instantiate model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",  # or whatever model you want
    temperature=0
)

In [14]:
# connect retriever to LLM
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [15]:
# query
rag_chain.invoke("What do the documents say about the effect of microplastics on humans?")

'The documents indicate that while adverse effects of microplastics on marine organisms are known, the health effects on humans are less well-studied. The discussion on health and diseases associated with microplastics is primarily based on in-vitro studies, small pilot human studies, and speculation from changes observed in marine organisms. Notable adverse effects identified include genotoxicity and cytotoxicity, with specific findings showing that microplastics can increase the frequency of micronucleation and other nuclear abnormalities in human blood lymphocytes. These effects have been linked to disorders such as infertility, diabetes, obesity, and cardiovascular disease. Additionally, the size of microplastics is critical, as smaller particles (less than 100 μm) can penetrate biological barriers and accumulate in tissues, including the placenta. While microplastics can be ingested or inhaled, the human health effects remain largely unknown, and limited data suggest potential imm

### Evaluate model

In [16]:
evaluator = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

def evaluate_answer(reference, prediction):
    prompt = f"""
    Reference Answer:
    {reference}

    Predicted Answer:
    {prediction}

    Score from 0 to 3:
    0 = Completely wrong
    1 = Mostly wrong
    2 = Mostly correct
    3 = Fully correct

    Only output the score.
    """
    return evaluator.predict(prompt)